<a href="https://colab.research.google.com/github/dvdjda/pipeline-maintenance-report-JUL-AUG-2024/blob/main/lesson1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd
train = pd.read_parquet("Dataset/train-product-review.parquet")
train.head()

,review_id,product_id,reviewer_id,stars,review_body,review_title,language,product_category
0,fr_0392802,product_fr_0863151,reviewer_fr_0127593,2,"Décevant, la qualité du son de ce vinyle est p...",moyen,fr,other
1,fr_0774965,product_fr_0516058,reviewer_fr_0836878,3,Les caractéristiques inscrites semblent identi...,produit reçu dans un autre emballage que sur l...,fr,musical_instruments
2,en_0364619,product_en_0917407,reviewer_en_0530306,5,Great smell and tasting tea. Love the box too.,Definitely a guest or anytime tea.,en,grocery
3,de_0437491,product_de_0932077,reviewer_de_0360484,4,Druck etwas pixelig. Sonst schickes Set,Schönes Set,de,toy
4,fr_0557442,product_fr_0993213,reviewer_fr_0009081,2,Le disque 6 était défectueux et j'ai perdu 15 ...,Un peu déçu,fr,other


In [14]:
train.shape

(20000, 8)

In [16]:
train['stars'].value_counts().soft_index

AttributeError: 'Series' object has no attribute 'soft_index'

In [17]:
train['language'].value_counts()

,count
language,
en,6736
de,6634
fr,6630


In [23]:
def stars_to_sentiment(stars):
  if stars <=2:
    return "Negative"
  elif stars ==3:
      return "Neutral"
  else: return "Positive"
train['sentiment'] = train['stars'].apply(stars_to_sentiment)
train['sentiment'].value_counts()

,count
sentiment,
Negative,8019
Positive,7975
Neutral,4006


In [24]:
train_en = train[train['language'] == 'en'].copy()
train_en.shape

(6736, 9)

In [27]:
print(train_en['review_body'].iloc[0])
print("---")
print("Sentiment:", train_en['sentiment'].iloc[0])

Great smell and tasting tea. Love the box too.
---
Sentiment: Positive


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features = 5000)
X = vectorizer.fit_transform(train_en['review_body'])
y = train_en['sentiment']
print(X.shape)

(6736, 5000)


In [29]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape, X_val.shape)

(5388, 5000) (1348, 5000)


In [30]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [31]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, predictions))

Accuracy: 0.6602373887240356
